# HPC Lab 8: GPU Programming with CUDA
**Orazgali Azamat, BDA-2402**

## Step 3: Check the GPU

In [1]:
!nvidia-smi

Tue Nov  3 10:07:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.54                 Driver Version: 595.79         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090 ...    On  |   00000000:02:00.0  On |                  N/A |
| N/A   53C    P8             24W /   60W |    1749MiB /  24463MiB |     14%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 4: Check the CUDA Compiler

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Part 1: Your First CUDA Kernel

In [3]:
%%writefile hello.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Hello from the GPU!\n");
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}


Overwriting hello.cu


In [4]:
!nvcc hello.cu -o hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [5]:
!./hello

Hello from the CPU!


Hello from the GPU!


## Part 2: Blocks and Threads

In [6]:
%%writefile blocks_threads.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Block %d, Thread %d\n",
           blockIdx.x,
           threadIdx.x);
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}


Overwriting blocks_threads.cu


In [7]:
!nvcc blocks_threads.cu -o blocks_threads

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [8]:
!./blocks_threads

Hello from the CPU!


Block 0, Thread 0
Block 0, Thread 1
Block 0, Thread 2
Block 0, Thread 3
Block 1, Thread 0
Block 1, Thread 1
Block 1, Thread 2
Block 1, Thread 3


## Part 3: Global Thread Index

In [9]:
%%writefile global_index.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void showIndex()
{
    int index =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    printf("Block %d, Thread %d, Global index %d\n",
           blockIdx.x,
           threadIdx.x,
           index);
}

int main()
{
    printf("Hello from the CPU!\n");

    showIndex<<<3, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}


Overwriting global_index.cu


In [10]:
!nvcc global_index.cu -o global_index

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [11]:
!./global_index

Hello from the CPU!


Block 0, Thread 0, Global index 0
Block 0, Thread 1, Global index 1
Block 0, Thread 2, Global index 2
Block 0, Thread 3, Global index 3
Block 2, Thread 0, Global index 8
Block 2, Thread 1, Global index 9
Block 2, Thread 2, Global index 10
Block 2, Thread 3, Global index 11
Block 1, Thread 0, Global index 4
Block 1, Thread 1, Global index 5
Block 1, Thread 2, Global index 6
Block 1, Thread 3, Global index 7


## Part 5: GPU Vector Addition

In [12]:
%%writefile vector_add.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    vectorAdd<<<blocks, threadsPerBlock>>>(
        dev_A, dev_B, dev_C, N);

    // Wait until the GPU finishes
    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("C[0] = %f\n", C[0]);
    printf("C[10] = %f\n", C[10]);
    printf("C[N-1] = %f\n", C[N-1]);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting vector_add.cu


In [13]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
!./vector_add

C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000


## Parts 6 and 7: Block Size and Kernel Execution Time
vector_add_timing.cu is vector_add.cu with the cudaEvent timing from Part 7. For each block size the line `int threadsPerBlock = 256;` is changed with sed, the program is recompiled and run 5 times. CUDA_MODULE_LOADING=EAGER makes the CUDA runtime load the kernel code when the program starts; with the default lazy loading the first kernel launch also includes the loading time.

In [15]:
%%writefile vector_add_timing.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    vectorAdd<<<blocks, threadsPerBlock>>>(
        dev_A, dev_B, dev_C, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float milliseconds = 0.0f;

    cudaEventElapsedTime(
        &milliseconds, start, stop);

    printf("Kernel time: %f ms\n",
           milliseconds);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    // Wait until the GPU finishes
    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("threadsPerBlock = %d, blocks = %d\n", threadsPerBlock, blocks);
    printf("C[0] = %f\n", C[0]);
    printf("C[10] = %f\n", C[10]);
    printf("C[N-1] = %f\n", C[N-1]);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting vector_add_timing.cu


In [16]:
import subprocess, re, os

os.environ["CUDA_MODULE_LOADING"] = "EAGER"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

N = 1000000
sizes = [32, 64, 128, 256, 512]
averages = {}
for t in sizes:
    sh(f"sed 's/int threadsPerBlock = 256;/int threadsPerBlock = {t};/' vector_add_timing.cu > vat_{t}.cu")
    sh(f"nvcc vat_{t}.cu -o vat_{t}")
    times = []
    for r in range(5):
        out = sh(f"./vat_{t}")
        times.append(float(re.search(r"Kernel time: ([0-9.]+)", out).group(1)))
    print(out.strip())
    print("runs (ms):", ", ".join(f"{x:.6f}" for x in times))
    averages[t] = sum(times) / len(times)
    print()

print("Threads per Block | Blocks | Average Kernel Time (ms)")
for t in sizes:
    print(f"{t:17d} | {(N + t - 1) // t:6d} | {averages[t]:.6f}")

Kernel time: 0.090112 ms
threadsPerBlock = 32, blocks = 31250
C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
runs (ms): 0.145472, 0.094400, 0.129568, 0.111872, 0.090112



Kernel time: 0.191584 ms
threadsPerBlock = 64, blocks = 15625
C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
runs (ms): 0.111008, 0.146688, 0.142560, 0.153408, 0.191584



Kernel time: 0.155136 ms
threadsPerBlock = 128, blocks = 7813
C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
runs (ms): 0.184640, 0.109408, 0.216384, 0.115808, 0.155136



Kernel time: 0.113184 ms
threadsPerBlock = 256, blocks = 3907
C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
runs (ms): 0.092672, 0.121120, 0.236640, 0.154560, 0.113184



Kernel time: 0.138496 ms
threadsPerBlock = 512, blocks = 1954
C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
runs (ms): 0.151968, 0.165536, 0.095712, 0.098496, 0.138496

Threads per Block | Blocks | Average Kernel Time (ms)
               32 |  31250 | 0.114285
               64 |  15625 | 0.149050
              128 |   7813 | 0.156275
              256 |   3907 | 0.143635
              512 |   1954 | 0.130042


## Part 8: CPU versus GPU
cpu_vs_gpu.cu runs the CPU loop (timed with std::chrono) and the GPU kernel (timed with cudaEvent). It is compiled with N = 10^6 and, with sed, with N = 10^7. Each version is run 5 times.

In [17]:
%%writefile cpu_vs_gpu.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);
    float *C_gpu = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    auto cpu_start = std::chrono::high_resolution_clock::now();

    for (int i = 0; i < N; ++i)
    {
        C[i] = A[i] + B[i];
    }

    auto cpu_end = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(cpu_end - cpu_start).count();

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    int threadsPerBlock = 256;

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    cudaEvent_t total_start, total_stop, start, stop;

    cudaEventCreate(&total_start);
    cudaEventCreate(&total_stop);
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(total_start);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    cudaEventRecord(start);

    vectorAdd<<<blocks, threadsPerBlock>>>(
        dev_A, dev_B, dev_C, N);

    cudaEventRecord(stop);

    cudaMemcpy(C_gpu, dev_C, bytes, cudaMemcpyDeviceToHost);

    cudaEventRecord(total_stop);

    cudaEventSynchronize(total_stop);

    float kernel_ms = 0.0f;
    float total_ms = 0.0f;

    cudaEventElapsedTime(&kernel_ms, start, stop);
    cudaEventElapsedTime(&total_ms, total_start, total_stop);

    int errors = 0;

    for (int i = 0; i < N; ++i)
    {
        if (C_gpu[i] != C[i])
        {
            errors++;
        }
    }

    printf("N = %d\n", N);
    printf("CPU time: %f ms\n", cpu_ms);
    printf("GPU kernel time: %f ms\n", kernel_ms);
    printf("GPU time with copies: %f ms\n", total_ms);
    printf("Mismatches: %d\n", errors);

    cudaEventDestroy(total_start);
    cudaEventDestroy(total_stop);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);
    free(C_gpu);

    return 0;
}


Overwriting cpu_vs_gpu.cu


In [18]:
sh("nvcc cpu_vs_gpu.cu -o cpu_vs_gpu_1e6")
sh("sed 's/const int N = 1000000;/const int N = 10000000;/' cpu_vs_gpu.cu > cpu_vs_gpu_1e7.cu")
sh("nvcc cpu_vs_gpu_1e7.cu -o cpu_vs_gpu_1e7")

for name in ["cpu_vs_gpu_1e6", "cpu_vs_gpu_1e7"]:
    cpu, ker, tot = [], [], []
    for r in range(5):
        out = sh("./" + name)
        cpu.append(float(re.search(r"CPU time: ([0-9.]+)", out).group(1)))
        ker.append(float(re.search(r"GPU kernel time: ([0-9.]+)", out).group(1)))
        tot.append(float(re.search(r"GPU time with copies: ([0-9.]+)", out).group(1)))
    print(out.strip())
    print("CPU runs (ms):        ", ", ".join(f"{x:.6f}" for x in cpu))
    print("GPU kernel runs (ms): ", ", ".join(f"{x:.6f}" for x in ker))
    print("GPU + copies runs (ms):", ", ".join(f"{x:.6f}" for x in tot))
    print(f"Average: CPU {sum(cpu)/5:.6f} ms, GPU kernel {sum(ker)/5:.6f} ms, GPU + copies {sum(tot)/5:.6f} ms")
    print()

N = 1000000
CPU time: 3.052751 ms
GPU kernel time: 0.042176 ms
GPU time with copies: 3.609664 ms
Mismatches: 0
CPU runs (ms):         2.324672, 2.321370, 2.755021, 2.373640, 3.052751
GPU kernel runs (ms):  0.214112, 0.018848, 0.044448, 0.085472, 0.042176
GPU + copies runs (ms): 3.698880, 4.030208, 2.903808, 4.584320, 3.609664
Average: CPU 2.565491 ms, GPU kernel 0.081011 ms, GPU + copies 3.765376 ms



N = 10000000
CPU time: 33.531689 ms
GPU kernel time: 0.445216 ms
GPU time with copies: 36.117790 ms
Mismatches: 0
CPU runs (ms):         26.283460, 24.052996, 25.365330, 24.452574, 33.531689
GPU kernel runs (ms):  0.300128, 1.177312, 0.295744, 0.265344, 0.445216
GPU + copies runs (ms): 40.581184, 32.719681, 32.907425, 37.994816, 36.117790
Average: CPU 26.737210 ms, GPU kernel 0.496749 ms, GPU + copies 36.064179 ms



## Part 10: Matrix Addition

In [19]:
%%writefile matrix_add.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void matrixAdd(
    const float *A,
    const float *B,
    float *C,
    int width,
    int height)
{
    int col =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    int row =
        blockIdx.y * blockDim.y +
        threadIdx.y;

    if (row < height && col < width)
    {
        int index = row * width + col;

        C[index] = A[index] + B[index];
    }
}

int main()
{
    const int width = 512;
    const int height = 512;

    size_t bytes = width * height * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int row = 0; row < height; ++row)
    {
        for (int col = 0; col < width; ++col)
        {
            A[row * width + col] = (float)(row + col);
            B[row * width + col] = (float)(2 * row - col);
        }
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);

    dim3 blocks(
        (width + threads.x - 1) / threads.x,
        (height + threads.y - 1) / threads.y);

    matrixAdd<<<blocks, threads>>>(
        dev_A,
        dev_B,
        dev_C,
        width,
        height);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("Grid: %d x %d blocks, block: %d x %d threads\n",
           blocks.x, blocks.y, threads.x, threads.y);

    int check[5][2] = {{0, 0}, {0, 511}, {10, 20}, {256, 100}, {511, 511}};

    for (int k = 0; k < 5; ++k)
    {
        int row = check[k][0];
        int col = check[k][1];
        printf("C[%d][%d] = %f (expected %f)\n",
               row, col, C[row * width + col], (float)(3 * row));
    }

    int errors = 0;

    for (int i = 0; i < width * height; ++i)
    {
        if (C[i] != A[i] + B[i])
        {
            errors++;
        }
    }

    printf("Mismatches: %d of %d elements\n", errors, width * height);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}


Overwriting matrix_add.cu


In [20]:
!nvcc matrix_add.cu -o matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [21]:
!./matrix_add

Grid: 32 x 32 blocks, block: 16 x 16 threads
C[0][0] = 0.000000 (expected 0.000000)
C[0][511] = 0.000000 (expected 0.000000)
C[10][20] = 30.000000 (expected 30.000000)
C[256][100] = 768.000000 (expected 768.000000)
C[511][511] = 1533.000000 (expected 1533.000000)
Mismatches: 0 of 262144 elements


## Optional Challenge: Matrix Multiplication
First a small 4 x 4 test (N changed with sed), then N = 512. Both are compared with the CPU result.

In [22]:
%%writefile matrix_mul.cu
#include <bits/stdc++.h>
#include <cuda_runtime.h>

__global__ void matrixMul(
    const float *A,
    const float *B,
    float *C,
    int N)
{
    int col =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    int row =
        blockIdx.y * blockDim.y +
        threadIdx.y;

    if (row < N && col < N)
    {
        float sum = 0.0f;

        for (int k = 0; k < N; ++k)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

int main()
{
    const int N = 512;

    size_t bytes = N * N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);
    float *C_cpu = (float *)malloc(bytes);

    for (int i = 0; i < N * N; ++i)
    {
        A[i] = (float)(i % 7);
        B[i] = (float)(i % 5);
    }

    for (int row = 0; row < N; ++row)
    {
        for (int col = 0; col < N; ++col)
        {
            float sum = 0.0f;

            for (int k = 0; k < N; ++k)
            {
                sum += A[row * N + k] * B[k * N + col];
            }

            C_cpu[row * N + col] = sum;
        }
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);

    dim3 blocks(
        (N + threads.x - 1) / threads.x,
        (N + threads.y - 1) / threads.y);

    matrixMul<<<blocks, threads>>>(
        dev_A,
        dev_B,
        dev_C,
        N);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    if (N <= 8)
    {
        for (int row = 0; row < N; ++row)
        {
            for (int col = 0; col < N; ++col)
            {
                printf("%8.1f", C[row * N + col]);
            }
            printf("\n");
        }
    }

    double max_diff = 0.0;

    for (int i = 0; i < N * N; ++i)
    {
        max_diff = std::max(max_diff, (double)fabs(C[i] - C_cpu[i]));
    }

    printf("N = %d\n", N);
    printf("C[0][0] = %f, CPU = %f\n", C[0], C_cpu[0]);
    printf("C[N-1][N-1] = %f, CPU = %f\n", C[N * N - 1], C_cpu[N * N - 1]);
    printf("Max difference GPU vs CPU: %f\n", max_diff);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);
    free(C_cpu);

    return 0;
}


Overwriting matrix_mul.cu


In [23]:
sh("sed 's/const int N = 512;/const int N = 4;/' matrix_mul.cu > matrix_mul_4.cu")
sh("nvcc matrix_mul_4.cu -o matrix_mul_4")
print(sh("./matrix_mul_4"))

    16.0    17.0    13.0     4.0
    38.0    28.0    13.0    28.0
    25.0    25.0    20.0    10.0
    26.0     8.0    20.0    27.0
N = 4
C[0][0] = 16.000000, CPU = 16.000000
C[N-1][N-1] = 27.000000, CPU = 27.000000
Max difference GPU vs CPU: 0.000000



In [24]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!./matrix_mul

N = 512
C[0][0] = 3053.000000, CPU = 3053.000000
C[N-1][N-1] = 3066.000000, CPU = 3066.000000
Max difference GPU vs CPU: 0.000000
